In [9]:
import pandas as pd
import numpy as np

# ============================================
# LOAD TRAINING DATA ONLY
# ============================================
train = pd.read_csv("../data/train.csv", index_col=0)
origin_date = pd.to_datetime(train["Date"]).min()


In [10]:
train.head()


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [11]:
def add_lag_features(df):
    df_features = df.copy()
    df_features["Date"] = pd.to_datetime(df_features["Date"])
    df_features = df_features.sort_values(
        ["Store ID", "Product ID", "Date"]
    ).reset_index(drop=True)

    group = df_features.groupby(["Store ID", "Product ID"], group_keys=False)

    def shifted_rolling(column, window, stat):
        shifted = group[column].shift(1)
        rolling = shifted.groupby(
            [df_features["Store ID"], df_features["Product ID"]]
        ).rolling(window, min_periods=window)

        if stat == "mean":
            values = rolling.mean()
        elif stat == "std":
            values = rolling.std()
        else:
            raise ValueError(f"Unsupported rolling stat: {stat}")

        return values.reset_index(level=[0, 1], drop=True)

    # =========================================================
    # 1. DEMAND (CORE SIGNAL)
    # =========================================================
    df_features["lag_demand_1"] = group["Demand"].shift(1)
    df_features["lag_demand_7"] = group["Demand"].shift(7)
    df_features["lag_demand_28"] = group["Demand"].shift(28)
    df_features["roll_demand_mean_7"] = shifted_rolling("Demand", 7, "mean")
    df_features["roll_demand_std_7"] = shifted_rolling("Demand", 7, "std")
    df_features["roll_demand_mean_28"] = shifted_rolling("Demand", 28, "mean")

    # =========================================================
    # 2. INVENTORY / SUPPLY
    # =========================================================
    df_features["lag_inventory_1"] = group["Inventory Level"].shift(1)
    df_features["roll_inventory_mean_7"] = shifted_rolling(
        "Inventory Level", 7, "mean"
    )
    df_features["lag_units_sold_1"] = group["Units Sold"].shift(1)
    df_features["lag_units_ordered_1"] = group["Units Ordered"].shift(1)

    # =========================================================
    # 3. PRICE / DISCOUNT / COMPETITION
    # =========================================================
    df_features["lag_price_1"] = group["Price"].shift(1)
    df_features["price_change_1"] = df_features["Price"] - df_features["lag_price_1"]
    df_features["price_gap"] = (
        df_features["Price"] - df_features["Competitor Pricing"]
    )
    df_features["lag_discount_1"] = group["Discount"].shift(1)
    df_features["roll_discount_mean_7"] = shifted_rolling("Discount", 7, "mean")
    df_features["lag_competitor_price_1"] = group["Competitor Pricing"].shift(1)

    # =========================================================
    # 4. EVENT FLAGS
    # =========================================================
    df_features["roll_promotion_7"] = shifted_rolling("Promotion", 7, "mean")
    df_features["roll_epidemic_28"] = shifted_rolling("Epidemic", 28, "mean")

    return df_features


train = add_lag_features(train)


In [12]:
train = train.dropna().reset_index(drop=True)
train.head()


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,lag_units_sold_1,lag_units_ordered_1,lag_price_1,price_change_1,price_gap,lag_discount_1,roll_discount_mean_7,lag_competitor_price_1,roll_promotion_7,roll_epidemic_28
0,2022-01-29,S001,P0001,Electronics,North,263,69,0,65.00,5,...,201.0,343.0,80.40,-15.40,-12.11,20.0,15.000000,73.36,0.571429,0.0
1,2022-01-30,S001,P0001,Electronics,North,537,73,0,71.59,0,...,69.0,0.0,65.00,6.59,1.90,5.0,15.714286,77.11,0.571429,0.0
2,2022-01-31,S001,P0001,Electronics,North,464,103,0,72.10,10,...,73.0,0.0,71.59,0.51,1.87,0.0,13.571429,69.69,0.428571,0.0
3,2022-02-01,S001,P0001,Electronics,North,361,102,0,70.53,5,...,103.0,0.0,72.10,-1.57,3.97,10.0,11.428571,70.23,0.285714,0.0
4,2022-02-02,S001,P0001,Electronics,North,259,94,215,60.16,20,...,102.0,0.0,70.53,-10.37,5.20,5.0,10.714286,66.56,0.285714,0.0


In [ ]:
# ============================================
# FIT ENCODING MAPS AFTER TRAIN LAGS EXIST
# ============================================
train_temp = train.copy()

store_freq_map = train_temp["Store ID"].value_counts().to_dict()
product_freq_map = train_temp["Product ID"].value_counts().to_dict()

store_product_freq_map = (
    train_temp["Store ID"].astype(str)
    + "_" +
    train_temp["Product ID"].astype(str)
).value_counts().to_dict()


# ============================================
# ENCODING FUNCTION
# ============================================
def encode_dataset(df):
    df_encoded = df.copy()

    # -----------------------
    # DATE FEATURES
    # -----------------------
    df_encoded["Date"] = pd.to_datetime(df_encoded["Date"])

    df_encoded["time_idx"] = (
        df_encoded["Date"] - origin_date
    ).dt.days

    month = df_encoded["Date"].dt.month
    dayofweek = df_encoded["Date"].dt.dayofweek

    df_encoded["is_weekend"] = (dayofweek >= 5).astype(int)

    df_encoded["month_sin"] = np.sin(2 * np.pi * month / 12)
    df_encoded["month_cos"] = np.cos(2 * np.pi * month / 12)

    df_encoded["dow_sin"] = np.sin(2 * np.pi * dayofweek / 7)
    df_encoded["dow_cos"] = np.cos(2 * np.pi * dayofweek / 7)

    # -----------------------
    # FREQUENCY ENCODING (TRAIN-BASED)
    # -----------------------
    df_encoded["store_freq"] = df_encoded["Store ID"].map(store_freq_map).fillna(0)
    df_encoded["product_freq"] = df_encoded["Product ID"].map(product_freq_map).fillna(0)

    store_product_key = (
        df_encoded["Store ID"].astype(str)
        + "_" +
        df_encoded["Product ID"].astype(str)
    )

    df_encoded["store_product_freq"] = (
        store_product_key.map(store_product_freq_map).fillna(0)
    )

    # -----------------------
    # DROP RAW COLUMNS
    # -----------------------
    df_encoded.drop(
        columns=["Date", "Store ID", "Product ID"],
        inplace=True
    )

    # -----------------------
    # ONE HOT ENCODING
    # -----------------------
    low_card_cols = df_encoded.select_dtypes(include="object").columns

    df_encoded = pd.get_dummies(
        df_encoded,
        columns=low_card_cols,
        dtype=int
    )

    return df_encoded


# ============================================
# ENCODE TRAIN ONLY
# ============================================
train_encoded = encode_dataset(train)
train_columns = train_encoded.columns.tolist()


def transform_for_prediction(df_with_lags):
    encoded = encode_dataset(df_with_lags)
    return encoded.reindex(columns=train_columns, fill_value=0)


# ============================================
# SAVE TRAIN FEATURES ONLY
# ============================================
train_encoded.to_csv("../data/transformed_train.csv", index=False)

print("Train shape:", train_encoded.shape)
